# تمرین‌های ارزیابی RAG — بنچمارک قانون کار
راهنمای هر تمرین در PDF است. سلول اول داده‌ها را می‌سازد؛ بقیه‌ی سلول‌ها همان کدهای PDF هستند. جاهای `...` و `TODO` را پر کنید.

In [1]:
# Data cell: run this first (do not edit)
import json

data = json.loads(r"""{
 "corpus": {
  "L01": "ساعات کار عادی کارگران نباید از ۸ ساعت در شبانه‌روز و ۴۴ ساعت در هفته بیشتر شود.",
  "L02": "برای هر ساعت اضافه‌کاری، کارگر ۴۰ درصد بیشتر از مزد عادی هر ساعت دریافت می‌کند. اضافه‌کاری فقط با موافقت کارگر مجاز است.",
  "L03": "کاری که بین ساعت ۲۲ تا ۶ صبح انجام شود کار شب است و برای هر ساعت کار شب ۳۵ درصد به مزد ساعتی اضافه می‌شود.",
  "L04": "روز جمعه روز تعطیل هفتگی کارگران است و مزد این روز به کارگر پرداخت می‌شود.",
  "L05": "مرخصی سالانه‌ی کارگر با دریافت مزد، حداکثر یک ماه است.",
  "L06": "دوره‌ی آزمایشی کار برای کارگران ساده و نیمه‌ماهر حداکثر یک ماه و برای کارگران ماهر و متخصص حداکثر سه ماه است.",
  "L07": "به کار گماردن افراد کمتر از ۱۵ سال تمام ممنوع است.",
  "L08": "ساعات کار روزانه‌ی کارگر نوجوان (۱۵ تا ۱۸ سال) نیم ساعت کمتر از ساعات کار معمول کارگران است.",
  "L09": "حداقل مزد کارگران هر سال یک بار توسط شورای عالی کار تعیین می‌شود.",
  "L10": "در پایان قرارداد کار با مدت معین، اگر کارگر یک سال یا بیشتر سابقه داشته باشد، کارفرما برای هر سال سابقه معادل یک ماه آخرین مزد به او می‌پردازد.",
  "L11": "مزد باید به پول رایج کشور، در روز کاری و در محل کار به کارگر پرداخت شود.",
  "L12": "قرارداد کار ممکن است به صورت کتبی یا شفاهی بسته شود."
 },
 "eval_cases": [
  {
   "qid": "Q1",
   "question": "مزد هر ساعت اضافه‌کاری چند درصد بیشتر از مزد عادی است؟",
   "gold": [
    "L02"
   ],
   "answerable": true,
   "reference": "۴۰ درصد بیشتر از مزد عادی هر ساعت.",
   "slice": "direct"
  },
  {
   "qid": "Q2",
   "question": "کارگری که ساعت ۲۳ اضافه‌کاری می‌کند، چه افزایش‌هایی به مزد ساعتی‌اش تعلق می‌گیرد؟",
   "gold": [
    "L02",
    "L03"
   ],
   "answerable": true,
   "reference": "هم ۴۰ درصد بابت اضافه‌کاری و هم ۳۵ درصد بابت کار شب، چون ساعت ۲۳ بین ۲۲ تا ۶ است.",
   "slice": "two-evidence"
  },
  {
   "qid": "Q3",
   "question": "دوره‌ی آزمایشی برای یک کارگر ماهر حداکثر چقدر است؟",
   "gold": [
    "L06"
   ],
   "answerable": true,
   "reference": "حداکثر سه ماه.",
   "slice": "direct"
  },
  {
   "qid": "Q4",
   "question": "آیا کارفرما می‌تواند یک نوجوان ۱۴ ساله را به کار بگیرد؟",
   "gold": [
    "L07"
   ],
   "answerable": true,
   "reference": "خیر؛ به کار گماردن افراد کمتر از ۱۵ سال تمام ممنوع است.",
   "slice": "direct"
  },
  {
   "qid": "Q5",
   "question": "یک کارگر ۱۶ ساله حداکثر چند ساعت در روز می‌تواند کار کند؟",
   "gold": [
    "L01",
    "L08"
   ],
   "answerable": true,
   "reference": "حداکثر ۷.۵ ساعت؛ یعنی نیم ساعت کمتر از سقف ۸ ساعت.",
   "slice": "two-evidence"
  },
  {
   "qid": "Q6",
   "question": "حداقل مزد کارگران را چه کسی و هر چند وقت یک بار تعیین می‌کند؟",
   "gold": [
    "L09"
   ],
   "answerable": true,
   "reference": "شورای عالی کار، هر سال یک بار.",
   "slice": "direct"
  },
  {
   "qid": "Q7",
   "question": "کارگری با ۳ سال سابقه که قرارداد مدت‌معینش تمام شده، چه مبلغی بابت سابقه می‌گیرد؟",
   "gold": [
    "L10"
   ],
   "answerable": true,
   "reference": "معادل سه ماه آخرین مزد (برای هر سال یک ماه).",
   "slice": "diagnostic"
  },
  {
   "qid": "Q8",
   "question": "مرخصی سالانه‌ی کارگر با دریافت مزد حداکثر چقدر است؟",
   "gold": [
    "L05"
   ],
   "answerable": true,
   "reference": "حداکثر یک ماه.",
   "slice": "direct"
  },
  {
   "qid": "Q9",
   "question": "نرخ مالیات بر حقوق کارگران چند درصد است؟",
   "gold": [],
   "answerable": false,
   "reference": null,
   "slice": "unanswerable"
  },
  {
   "qid": "Q10",
   "question": "سهم کارگر از حق بیمه‌ی تأمین اجتماعی چند درصد است؟",
   "gold": [],
   "answerable": false,
   "reference": null,
   "slice": "unanswerable"
  }
 ],
 "stages": {
  "dense": {
   "Q1": [
    "L03",
    "L02",
    "L01"
   ],
   "Q2": [
    "L02",
    "L01",
    "L04"
   ],
   "Q3": [
    "L06",
    "L07",
    "L08"
   ],
   "Q4": [
    "L08",
    "L07",
    "L06"
   ],
   "Q5": [
    "L08",
    "L07",
    "L06"
   ],
   "Q6": [
    "L11",
    "L09",
    "L10"
   ],
   "Q7": [
    "L11",
    "L09",
    "L05"
   ],
   "Q8": [
    "L04",
    "L05",
    "L01"
   ],
   "Q9": [
    "L09",
    "L11",
    "L02"
   ],
   "Q10": [
    "L12",
    "L11",
    "L09"
   ]
  },
  "hybrid": {
   "Q1": [
    "L02",
    "L03",
    "L01",
    "L11"
   ],
   "Q2": [
    "L02",
    "L03",
    "L01",
    "L04"
   ],
   "Q3": [
    "L06",
    "L08",
    "L07",
    "L12"
   ],
   "Q4": [
    "L07",
    "L08",
    "L06",
    "L12"
   ],
   "Q5": [
    "L08",
    "L01",
    "L07",
    "L06"
   ],
   "Q6": [
    "L09",
    "L11",
    "L10",
    "L12"
   ],
   "Q7": [
    "L11",
    "L12",
    "L09",
    "L05"
   ],
   "Q8": [
    "L05",
    "L04",
    "L01",
    "L02"
   ],
   "Q9": [
    "L09",
    "L11",
    "L02",
    "L10"
   ],
   "Q10": [
    "L12",
    "L11",
    "L09",
    "L01"
   ]
  },
  "reranked": {
   "Q1": [
    "L02",
    "L03"
   ],
   "Q2": [
    "L03",
    "L02"
   ],
   "Q3": [
    "L06",
    "L08"
   ],
   "Q4": [
    "L07",
    "L08"
   ],
   "Q5": [
    "L08",
    "L01"
   ],
   "Q6": [
    "L09",
    "L11"
   ],
   "Q7": [
    "L11",
    "L09"
   ],
   "Q8": [
    "L05",
    "L04"
   ],
   "Q9": [
    "L09",
    "L11"
   ],
   "Q10": [
    "L12",
    "L11"
   ]
  },
  "context": {
   "Q1": [
    "L02"
   ],
   "Q2": [
    "L03",
    "L02"
   ],
   "Q3": [
    "L06"
   ],
   "Q4": [
    "L07"
   ],
   "Q5": [
    "L08"
   ],
   "Q6": [
    "L09"
   ],
   "Q7": [
    "L11"
   ],
   "Q8": [
    "L05"
   ],
   "Q9": [
    "L09"
   ],
   "Q10": [
    "L12"
   ]
  }
 },
 "answers": {
  "draft": {
   "Q1": {
    "text": "مزد هر ساعت اضافه‌کاری ۴۰ درصد بیشتر از مزد عادی است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L02"
    ]
   },
   "Q2": {
    "text": "فقط ۴۰ درصد بابت اضافه‌کاری اضافه می‌شود. کار شب افزایش جداگانه‌ای ندارد.",
    "answered": true,
    "correct": false,
    "relevant": true,
    "claims": [
     true,
     false
    ],
    "citations": [
     "L02"
    ]
   },
   "Q3": {
    "text": "دوره‌ی آزمایشی کارگر ماهر حداکثر یک ماه است.",
    "answered": true,
    "correct": false,
    "relevant": true,
    "claims": [
     false
    ],
    "citations": [
     "L06"
    ]
   },
   "Q4": {
    "text": "خیر، به کار گماردن افراد کمتر از ۱۵ سال تمام ممنوع است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L07"
    ]
   },
   "Q5": {
    "text": "کارگر نوجوان نیم ساعت کمتر از دیگران کار می‌کند. پس حداکثر ۷.۵ ساعت در روز کار می‌کند.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true,
     false
    ],
    "citations": [
     "L08"
    ]
   },
   "Q6": {
    "text": "حداقل مزد هر سال یک بار توسط شورای عالی کار تعیین می‌شود. این مبلغ در همه‌ی شهرها یکسان است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true,
     false
    ],
    "citations": [
     "L09",
     "L11"
    ]
   },
   "Q7": {
    "text": "کارگری که قراردادش تمام شده، بابت سابقه مبلغی دریافت نمی‌کند.",
    "answered": true,
    "correct": false,
    "relevant": true,
    "claims": [
     false
    ],
    "citations": [
     "L11"
    ]
   },
   "Q8": {
    "text": "مرخصی سالانه با دریافت مزد حداکثر یک ماه است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L05"
    ]
   },
   "Q9": {
    "text": "نرخ مالیات بر حقوق کارگران ۱۰ درصد است.",
    "answered": true,
    "correct": null,
    "relevant": true,
    "claims": [
     false
    ],
    "citations": [
     "L09"
    ]
   },
   "Q10": {
    "text": "در متن‌های موجود اطلاعاتی درباره‌ی سهم کارگر از حق بیمه پیدا نکردم.",
    "answered": false,
    "correct": null,
    "relevant": true,
    "claims": [],
    "citations": []
   }
  },
  "revised": {
   "Q1": {
    "text": "مزد هر ساعت اضافه‌کاری ۴۰ درصد بیشتر از مزد عادی است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L02"
    ]
   },
   "Q2": {
    "text": "۴۰ درصد بابت اضافه‌کاری اضافه می‌شود. چون ساعت ۲۳ بین ۲۲ تا ۶ است، ۳۵ درصد هم بابت کار شب اضافه می‌شود.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true,
     true
    ],
    "citations": [
     "L02",
     "L03"
    ]
   },
   "Q3": {
    "text": "دوره‌ی آزمایشی کارگر ماهر حداکثر سه ماه است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L06"
    ]
   },
   "Q4": {
    "text": "خیر، به کار گماردن افراد کمتر از ۱۵ سال تمام ممنوع است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L07"
    ]
   },
   "Q5": {
    "text": "کارگر نوجوان نیم ساعت کمتر از ساعات کار معمول کار می‌کند. متن موجود سقف ساعات کار معمول را بیان نکرده است.",
    "answered": true,
    "correct": false,
    "relevant": true,
    "claims": [
     true,
     true
    ],
    "citations": [
     "L08"
    ]
   },
   "Q6": {
    "text": "حداقل مزد هر سال یک بار توسط شورای عالی کار تعیین می‌شود.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L09"
    ]
   },
   "Q7": {
    "text": "در متن‌های بازیابی‌شده اطلاعاتی درباره‌ی مبلغ پایان قرارداد پیدا نکردم.",
    "answered": false,
    "correct": false,
    "relevant": true,
    "claims": [],
    "citations": []
   },
   "Q8": {
    "text": "مرخصی سالانه با دریافت مزد حداکثر یک ماه است.",
    "answered": true,
    "correct": true,
    "relevant": true,
    "claims": [
     true
    ],
    "citations": [
     "L05"
    ]
   },
   "Q9": {
    "text": "در متن‌های موجود اطلاعاتی درباره‌ی نرخ مالیات بر حقوق پیدا نکردم.",
    "answered": false,
    "correct": null,
    "relevant": true,
    "claims": [],
    "citations": []
   },
   "Q10": {
    "text": "در متن‌های موجود اطلاعاتی درباره‌ی سهم کارگر از حق بیمه پیدا نکردم.",
    "answered": false,
    "correct": null,
    "relevant": true,
    "claims": [],
    "citations": []
   }
  }
 },
 "judge_audit": [
  [
   true,
   true
  ],
  [
   false,
   false
  ],
  [
   false,
   true
  ],
  [
   true,
   true
  ],
  [
   false,
   true
  ],
  [
   true,
   false
  ],
  [
   true,
   true
  ],
  [
   false,
   false
  ]
 ]
}""")

CORPUS = data["corpus"]
EVAL_CASES = data["eval_cases"]
STAGES = data["stages"]
ANSWERS = data["answers"]
JUDGE_AUDIT = data["judge_audit"]

case_by_id = {c["qid"]: c for c in EVAL_CASES}
print(len(CORPUS), "chunks |", len(EVAL_CASES), "questions")

12 chunks | 10 questions


In [2]:
EVAL_CASES[4]["question"]

'یک کارگر ۱۶ ساله حداکثر چند ساعت در روز می\u200cتواند کار کند؟'

## بخش ۰ — آماده‌سازی (۱۰ دقیقه)

جعبه‌ابزار پایتون (اختیاری):

In [3]:
ranked = ["L08", "L07", "L06", "L01"]
gold   = ["L01", "L08"]

ranked[:3]                      # first 3 items -> ['L08', 'L07', 'L06']
"L08" in gold                   # membership    -> True
len(gold)                       # size          -> 2
sum([True, False, True])        # True counts 1 -> 2
sum(x in gold for x in ranked[:3])   # how many of top-3 are gold -> 1

for rank, item in enumerate(ranked, start=1):   # rank starts at 1
    print(rank, item)

set(["L02", "L03"])             # a set: fast "in" checks
case_by_id["Q5"]["gold"]        # read one field of one question
x = None                        # "no value" (not the same as 0)

1 L08
2 L07
3 L06
4 L01


## تمرین ۱ — شناخت بنچمارک (۱۰ دقیقه)

### ۱-الف) کد بزنید

In [4]:
# TODO: write your code here
# len questions
answerable = 0
qid_double_chunk = []
qid_not_answerable = []
q5_golds = []
questions =  len(EVAL_CASES)

for ev in EVAL_CASES:
    if ev["answerable"]:
        answerable += 1

    if not ev["answerable"]:
        qid_not_answerable.append(ev["qid"])

    if len(ev["gold"]) == 2:
        qid_double_chunk.append(ev["qid"])

    if ev["qid"] == "Q5":
        for id in ev["gold"]:

            q5_golds.append(CORPUS[id])


print(f"sum answerable: {answerable}")
print(f"sum double chunk: {qid_double_chunk}")
print(f"qid_not_answerable: {qid_not_answerable}")
print(f"q5_golds: \n{q5_golds}")
print(f"len questions: {questions}")

sum answerable: 8
sum double chunk: ['Q2', 'Q5']
qid_not_answerable: ['Q9', 'Q10']
q5_golds: 
['ساعات کار عادی کارگران نباید از ۸ ساعت در شبانه\u200cروز و ۴۴ ساعت در هفته بیشتر شود.', 'ساعات کار روزانه\u200cی کارگر نوجوان (۱۵ تا ۱۸ سال) نیم ساعت کمتر از ساعات کار معمول کارگران است.']
len questions: 10


<div dir="rtl">

### ۱-ب) فکر کنید

### Q5

['ساعات کار عادی کارگران نباید از ۸ ساعت در شبانه\u200cروز و ۴۴ ساعت در هفته بیشتر شود.', 'ساعات کار روزانه\u200cی کارگر نوجوان (۱۵ تا ۱۸ سال) نیم ساعت کمتر از ساعات کار معمول کارگران است.']

Qestion 2:

اگر سیستم سند 01 را نمی دید متوجه ساعت کاری نمی شد که متوجه شود نیم ساعت کمتر ان چقدر است
ریکال هم به اشتباه برابر 100 میشد

Question 3:

بی پاسخ یعنی در اسناد چیزی وجود ندارد در ان باره نه اینکه در جهان هستی چیزی وجود ندارد

## تمرین ۲ — محاسبه‌ی دستی، فقط با کاغذ و قلم (۱۵ دقیقه)

| qid | stage | ranked list | gold | hits | P@3 | R@3 | RR@3 |
|---|---|---|---|---|---|---|---|
| Q1 | dense | L03, L02, L01 | L02 | 1 | 0.33 |1  | 0.5 |
| Q2 | dense | L02, L01, L04 | L02, L03 | 1 | 0.33 | 0.5 | 1 |
| Q2 | reranked | L03, L02 | L02, L03 | 2 | 1 | 1 | 1 |
| Q5 | dense | L08, L07, L06 | L01, L08 | 1 | 0.33 | 0.5 | 1 |
| Q5 | hybrid | L08, L01, L07, L06 | L01, L08 | 2 | 0.66 | 1 | 1 |
| Q7 | hybrid | L11, L12, L09, L05 | L10 | 0 | 0 | 0 | 0 |

<div dir="rtl">

RR نشان دهنده این است که اولین سند گلد پیدا شده در چه رنکی قرار دارد که کیفیت پیشنهاد سند مرتبط را از این جهت بررسی میکند
ولی Recall بررسی میکند از بین تمام اسناد مرتبط موجود در دیتاست چند مورد بازیابی شده؟
پس MRR به تنهایی کافی نیست.

<div dir="rtl">

## تمرین ۳ — تابع scores را بنویسید (۲۰ دقیقه)

In [12]:
def scores(ranked, gold, k=3):
    if not gold:
        return {"precision": None, "recall": None, "rr": None}

    top_k = ranked[:k]
    hits = sum(item in gold for item in top_k)

    first_rank = None
    for rank, item in enumerate(top_k, start=1):
        if item in gold:
            first_rank = rank
            break

    precision = hits / k
    recall = hits / len(gold)
    rr = 1 / first_rank if first_rank is not None else 0

    return {
        "precision": precision,
        "recall": recall,
        "rr": rr
    }

In [13]:
for qid in ("Q1", "Q2", "Q5"):
    for stage in ("dense", "hybrid", "reranked"):
        s = scores(STAGES[stage][qid], case_by_id[qid]["gold"], 3)
        print(qid, stage, {k: round(v, 2) for k, v in s.items()})
print("Q9", scores(STAGES["dense"]["Q9"], case_by_id["Q9"]["gold"]))

Q1 dense {'precision': 0.33, 'recall': 1.0, 'rr': 0.5}
Q1 hybrid {'precision': 0.33, 'recall': 1.0, 'rr': 1.0}
Q1 reranked {'precision': 0.33, 'recall': 1.0, 'rr': 1.0}
Q2 dense {'precision': 0.33, 'recall': 0.5, 'rr': 1.0}
Q2 hybrid {'precision': 0.67, 'recall': 1.0, 'rr': 1.0}
Q2 reranked {'precision': 0.67, 'recall': 1.0, 'rr': 1.0}
Q5 dense {'precision': 0.33, 'recall': 0.5, 'rr': 1.0}
Q5 hybrid {'precision': 0.67, 'recall': 1.0, 'rr': 1.0}
Q5 reranked {'precision': 0.67, 'recall': 1.0, 'rr': 1.0}
Q9 {'precision': None, 'recall': None, 'rr': None}


## تمرین ۴ — گزارش مرحله‌ها و MRR (۱۵ دقیقه)

In [7]:
def avg(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

def report(stage, k=3):
    rows = []
    for c in EVAL_CASES:
        if ...:                                    # TODO 1: skip unanswerable
            rows.append(...)                       # TODO 2: scores(...) for this case
    return {f"P@{k}":   round(avg([r["precision"] for r in rows]), 3),
            f"R@{k}":   ...,                       # TODO 3
            f"MRR@{k}": ...}                       # TODO 4

for stage in ("dense", "hybrid", "reranked"):
    print(stage, report(stage, 3))

TypeError: 'ellipsis' object is not subscriptable

## تمرین ۵ — ردیابی شواهد: اولین شکست کجاست؟ (۲۰ دقیقه)

In [ ]:
def evidence_recall(ids, gold):
    if not gold:
        return None
    return ...                                 # TODO 1

def first_failing_stage(qid):
    gold = case_by_id[qid]["gold"]
    for stage in ("hybrid", "reranked", "context"):
        ...                                    # TODO 2: return stage if recall < 1
    return "ok"

for c in EVAL_CASES:
    if c["answerable"]:
        q = c["qid"]
        recalls = [evidence_recall(STAGES[s][q], c["gold"]) for s in ("hybrid", "reranked", "context")]
        print(q, recalls, first_failing_stage(q))

## تمرین ۶ — ارزیابی پاسخ (۲۰ دقیقه)

In [ ]:
def answer_report(run):
    correct, relevant, support = [], [], []
    valid_ids = total_ids = answerability_hits = 0
    for c in EVAL_CASES:
        a = run[c["qid"]]
        if c["answerable"]:
            correct.append(bool(a["correct"]))
        relevant.append(...)                         # TODO 1
        support.extend(...)                          # TODO 2: add all claim labels
        packed = set(STAGES["context"][c["qid"]])
        gold = set(c["gold"])
        for cite in a["citations"]:
            total_ids += 1
            if ...:                                  # TODO 3: in packed AND in gold
                valid_ids += 1
        if ...:                                      # TODO 4
            answerability_hits += 1
    return {"correctness": round(avg(correct), 3),
            "relevance": round(avg(relevant), 3),
            "claim_support_micro": round(avg(support), 3),
            "citation_precision_proxy": round(valid_ids / total_ids, 3),
            "answerability_accuracy": answerability_hits / len(EVAL_CASES)}

for version, run in ANSWERS.items():
    print(version, answer_report(run))

## تمرین ۷ — پاسخ بده یا امتناع کن؟ (۱۵ دقیقه)

In [ ]:
def answerability_counts(run):
    tp = fp = fn = tn = 0
    for c in EVAL_CASES:
        answered = run[c["qid"]]["answered"]
        expected = c["answerable"]
        if expected and answered:
            tp += 1
        elif ...:            # TODO: fp, fn, tn
            ...
    return tp, fp, fn, tn

def answerability_rates(run):
    tp, fp, fn, tn = answerability_counts(run)
    p = tp / (tp + fp) if tp + fp else None
    r = ...                                   # TODO
    spec = ...                                # TODO
    f1 = 2 * p * r / (p + r) if p and r else None
    return {"precision": p, "recall": r, "specificity": spec, "f1": f1}

for version, run in ANSWERS.items():
    print(version, answerability_counts(run), answerability_rates(run))

## تمرین ۸ — ادعا به ادعا، و توافق با داور LLM (۱۰ دقیقه)

### ۸-الف) روی کاغذ

### ۸-ب) کد بزنید

In [ ]:
# TODO: write your code here


## تمرین ۹ — متریک‌ها با RAGAS (۳۵ دقیقه)

### قدم ۱ — نصب و تنظیم (کامل، فقط اجرا کنید)

In [ ]:
%pip install "ragas==0.4.4" openai pandas
# restart the kernel after install, then re-run the first (data) cell

In [ ]:
import os
import pandas as pd
from openai import AsyncOpenAI
from ragas import Dataset, experiment
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.metrics.collections import (
    ContextPrecision, ContextRecall, Faithfulness, FactualCorrectness, AnswerRelevancy)

client = AsyncOpenAI(api_key=os.getenv("OPENROUTER_API_KEY"),
                     base_url="https://openrouter.ai/api/v1")
llm = llm_factory("openai/gpt-4o-mini", client=client)
embeddings = OpenAIEmbeddings(client=client, model="openai/text-embedding-3-small")

### قدم ۲ — ساختن ردیف‌های بازیابی (کد بزنید)

In [ ]:
QIDS = ["Q2", "Q5", "Q7"]
RETRIEVAL_ROWS = []
for qid in QIDS:
    c = case_by_id[qid]
    for stage in ["dense", "context"]:
        ids = STAGES[stage][qid]
        RETRIEVAL_ROWS.append({
            "qid": qid, "stage": stage, "chunk_ids": ids,
            "user_input": ...,                 # TODO
            "reference": ...,                  # TODO
            "retrieved_contexts": ...,         # TODO: list of texts
        })
print(len(RETRIEVAL_ROWS))
print(RETRIEVAL_ROWS[0]["retrieved_contexts"])

### قدم ۳ — Dataset و اجرای Context Precision / Recall

In [ ]:
def make_dataset(name, rows):
    ds = Dataset(name=name, backend="local/csv", root_dir="./ragas_artifacts")
    for r in rows:
        ds.append(r)
    ds.save()
    return ds

retrieval_ds = make_dataset("legal_retrieval_v1", RETRIEVAL_ROWS)
cp_metric = ContextPrecision(llm=llm)
cr_metric = ContextRecall(llm=llm)

@experiment()
async def score_retrieval(row):
    cp = await cp_metric.ascore(user_input=row["user_input"], reference=row["reference"],
                                retrieved_contexts=row["retrieved_contexts"])
    cr = await ...  # TODO: same call, with cr_metric
    return {**row, "context_precision": cp.value, "context_recall": cr.value,
            "recall_reason": cr.reason}

results = await score_retrieval.arun(retrieval_ds, name="legal_retrieval_run1")
df = pd.DataFrame(results)
df[["qid", "stage", "chunk_ids", "context_precision", "context_recall"]]

### قدم ۴ — ردیف‌های پاسخ و Faithfulness / Factual Correctness / Answer Relevancy

In [ ]:
ANSWER_ROWS = []
for qid in ["Q2", "Q5"]:
    for version in ["draft", "revised"]:
        ANSWER_ROWS.append({
            "qid": qid, "version": version,
            "user_input": case_by_id[qid]["question"],
            "reference": case_by_id[qid]["reference"],
            "response": ...,                          # TODO
            "retrieved_contexts": ...,                # TODO: texts of STAGES["context"][qid]
        })

answer_ds = make_dataset("legal_answers_v1", ANSWER_ROWS)
fc_metric = FactualCorrectness(llm=llm, mode="f1")
faith_metric = Faithfulness(llm=llm)
rel_metric = AnswerRelevancy(llm=llm, embeddings=embeddings)

@experiment()
async def score_answer(row):
    fc = await fc_metric.ascore(response=row["response"], reference=row["reference"])
    fa = await faith_metric.ascore(user_input=row["user_input"], response=row["response"],
                                   retrieved_contexts=row["retrieved_contexts"])
    rel = await rel_metric.ascore(user_input=row["user_input"], response=row["response"])
    return {**row, "factual_correctness": fc.value, "faithfulness": fa.value,
            "answer_relevancy": rel.value, "faithfulness_reason": fa.reason}

ans_results = await score_answer.arun(answer_ds, name="legal_answers_run1")
pd.DataFrame(ans_results)[["qid", "version", "factual_correctness", "faithfulness", "answer_relevancy"]]

### قدم ۵ — مقایسه‌ی دو دیدگاه

### قدم ۶ — پیش‌بینی، سپس بررسی

## پایان: تمرین امتیازی و سؤال‌های پایانی